# 作文を自動で採点する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/28_gec_education/essay_scoring.ipynb)

NLP クックブック 28-21 ／ 文法誤り訂正と言語教育

学習者の作文に、人が付けるのと近い点数を自動で付けたい。特徴量による回帰と文埋め込みによる回帰を比べ、採点者間の一致度と同じ指標 (QWK) で評価する。

In [1]:
!pip install -q datasets sentence-transformers

In [2]:
import re
import numpy as np
import pandas as pd
from datasets import load_dataset

df = load_dataset("nlpatunt/D_ASAP-AES", split="train").to_pandas()      # ASAP: 米国の生徒が書いた英語の作文と、人が付けた点数
print(df.columns.tolist(), len(df))
print(df.groupby("essay_set")["domain1_score"].agg(["count", "min", "max", "mean"]).round(2))

['essay_id', 'essay_set', 'essay', 'rater1_domain1', 'rater2_domain1', 'rater3_domain1', 'domain1_score', 'rater1_domain2', 'rater2_domain2', 'domain2_score', 'rater1_trait1', 'rater1_trait2', 'rater1_trait3', 'rater1_trait4', 'rater1_trait5', 'rater1_trait6', 'rater2_trait1', 'rater2_trait2', 'rater2_trait3', 'rater2_trait4', 'rater2_trait5', 'rater2_trait6', 'rater3_trait1', 'rater3_trait2', 'rater3_trait3', 'rater3_trait4', 'rater3_trait5', 'rater3_trait6', 'prompt_rubric_source_text_source', 'essay_set_descritions_source', 'general_guidelines_source'] 9083
           count  min  max   mean
essay_set                        
1           1248    2   12   8.52
2           1260    1    6   3.41
3           1208    0    3   1.85
4           1239    0    3   1.44
5           1264    0    4   2.44
6           1260    0    4   2.72
7           1098    2   24  16.07
8            506   10   55  36.73


In [3]:
essays = df[df["essay_set"] == 1].dropna(subset=["domain1_score"]).sample(frac=1, random_state=0).reset_index(drop=True)     # 課題 1 (意見文、2〜12 点)
essays["score"] = essays["domain1_score"].astype(int)
train, test = essays.iloc[:1000], essays.iloc[1000:]
print(len(train), "件で学習 /", len(test), "件で評価")

def handcrafted(text):
    words = re.findall(r"[A-Za-z']+", text.lower())
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    return [len(words), np.log(len(words) + 1), len(set(words)) / (len(words) + 1), np.mean([len(w) for w in words] or [0]),
            len(sentences), len(words) / (len(sentences) + 1), text.count(","), sum(len(w) >= 8 for w in words) / (len(words) + 1)]

names = ["語数", "log 語数", "異なり語の割合", "平均語長", "文数", "平均文長", "コンマの数", "長い語の割合"]
X_train, X_test = np.array([handcrafted(t) for t in train["essay"]]), np.array([handcrafted(t) for t in test["essay"]])
print(pd.Series({n: np.corrcoef(X_train[:, i], train["score"])[0, 1] for i, n in enumerate(names)}).round(3).to_string())

1000 件で学習 / 248 件で評価
語数         0.790
log 語数     0.811
異なり語の割合   -0.354
平均語長       0.293
文数         0.694
平均文長      -0.085
コンマの数      0.617
長い語の割合     0.276


In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import cohen_kappa_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def qwk(y_true, y_pred):
    """重み付き κ 係数 (quadratic weighted kappa): 点数の差が大きい食い違いほど重く罰する。作文採点の標準的な指標"""
    y_pred = np.clip(np.rint(y_pred), y_true.min(), y_true.max()).astype(int)
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

embedder = SentenceTransformer("all-MiniLM-L6-v2")
E_train = embedder.encode(train["essay"].tolist(), batch_size=32, normalize_embeddings=True)
E_test = embedder.encode(test["essay"].tolist(), batch_size=32, normalize_embeddings=True)

y_train, y_test = train["score"].to_numpy(), test["score"].to_numpy()
models = {
    "語数だけ": (X_train[:, :2], X_test[:, :2]),
    "手作りの特徴量": (X_train, X_test),
    "文埋め込み": (E_train, E_test),
    "特徴量 + 文埋め込み": (np.hstack([X_train, E_train * 10]), np.hstack([X_test, E_test * 10])),
}
for label, (a, b) in models.items():
    pred = make_pipeline(StandardScaler(), Ridge(alpha=10.0)).fit(a, y_train).predict(b)
    print(f"{label}: QWK = {qwk(y_test, pred):.3f} / 平均絶対誤差 = {np.abs(pred - y_test).mean():.2f}")

human = essays.dropna(subset=["rater1_domain1", "rater2_domain1"])
print("参考: 人の採点者 2 人の間の QWK =", round(cohen_kappa_score(human["rater1_domain1"].astype(int), human["rater2_domain1"].astype(int), weights="quadratic"), 3))

語数だけ: QWK = 0.793 / 平均絶対誤差 = 0.69
手作りの特徴量: QWK = 0.814 / 平均絶対誤差 = 0.62
文埋め込み: QWK = 0.550 / 平均絶対誤差 = 1.02
特徴量 + 文埋め込み: QWK = 0.740 / 平均絶対誤差 = 0.75
参考: 人の採点者 2 人の間の QWK = 0.704


- 作文の長さだけで、点数のかなりの部分が説明できてしまう。新しい手法は、必ず「語数だけ」のベースラインと比べる。長さだけを見るモデルは、意味のない長文に高得点を付けてしまう。
- この課題の点数は 2 人の採点者の点数の合計なので、1 人の採点者どうしの一致度 (QWK 0.70) より、モデルと合計点の一致度のほうが高く出る。人と同等以上に見えても、比べている対象が違う点に注意する。
- 文埋め込みは、この課題では手作りの特徴量より大きく劣った。一般的な文埋め込みは文章の話題を表すもので、文章の巧拙はあまり反映しない。
- 評価には QWK（重み付き κ）を使う。人の採点者どうしの一致度と同じ物差しなので、「人と同じくらい一致している」かどうかを直接比べられる。
- 課題（プロンプト）ごとに採点基準も点数の範囲も違う。ある課題で学習したモデルは、別の課題にはそのまま使えない。
- 自動採点は、内容の正しさや論理の一貫性をほとんど見ていない。重要な判定に使うときは、人の採点と併用し、極端な入力（丸写し、無関係な文章）への挙動を確かめる。
- 日本語学習者の作文では、文法誤りの数、語彙の難易度（この章の語彙の難しさのレシピ）、接続表現の使い方などが特徴量としてよく使われる。観点別（内容・構成・言語）に採点する方法もある。